# Browse eval responses

Look up the prompts a particular checkpoint gave to a particular eval.

Three indexing dimensions:
- **model** -- the underlying LLM (e.g. `meta-llama-Llama-3.1-8B-Instruct`,
  `Qwen-Qwen3-8B-Base`). Each model has its own `finetuned_scores_<model>.json`.
- **pole** -- the propensity this checkpoint was finetuned toward
  (`agreeableness-plus`, `effort-minus`, `narcissism-plus`, ...). Pole
  `base` means the unfinetuned model. The pole is the thing meant to push
  the eval score up or down -- e.g. an `agreeableness-plus` checkpoint
  should score high on the `agreeableness` eval, and `agreeableness-minus`
  should score low.
- **eval** -- which propensity is being *measured* (`effort`, `narcissism`,
  `agreeableness`, ...). The judge LLM scores each answer along this axis.

On-diagonal cells (pole `X-plus`/`X-minus` x eval `X`) are direct
elicitations; off-diagonal cells are cross-elicitations.

All `finetuned_scores_*.json` files in `../results/` are loaded into `SCORES`
(keyed by model). `get_responses` and `get_scores` stream the per-cell
`rows.jsonl` on demand instead of bundling every prompt + answer into
one giant JSON file.

Re-running `summarize_FT.py` does **not** overwrite this notebook --
delete it first if you want the fresh scaffold.


The `get_responses` function is the key function here - use it to look at all the conversations that were judged by gpt-5.4-mini (default judge model), or use `get_scores` to look at the scores that gpt-5.4-mini gave.

In [ ]:
import json
from pathlib import Path

_here = Path('.').resolve()
RESULTS_DIR = _here if _here.name == 'results' else _here / 'results'
EVAL_ROOT = (RESULTS_DIR.parent / 'eval_results' / 'finetuning').resolve()

SCORES = {}
for p in sorted(RESULTS_DIR.glob('finetuned_scores_*.json')):
    doc = json.loads(p.read_text())
    if doc.get('n_cells', 0) > 0:
        SCORES[doc['base_model']] = doc

print('Loaded scores for models:')
for m, doc in SCORES.items():
    print(f"  {m}  ({doc['n_cells']} cells across {doc['n_poles']} poles)")
print()
print('Use get_responses(model, pole, eval) and get_scores(model, pole, eval).')


In [ ]:
def _cell(model, pole, eval_propensity):
    if model not in SCORES:
        raise KeyError(f'unknown model {model!r}; loaded: {sorted(SCORES)}')
    cells = SCORES[model]['cells']
    if pole not in cells:
        raise KeyError(
            f'unknown pole {pole!r} for model {model!r}; '
            f'available: {sorted(cells)}'
        )
    if eval_propensity not in cells[pole]:
        raise KeyError(
            f'no eval {eval_propensity!r} for pole {pole!r}; '
            f'available: {sorted(cells[pole])}'
        )
    return cells[pole][eval_propensity]


def _iter_rows(model, pole, eval_propensity):
    cell = _cell(model, pole, eval_propensity)
    rows_path = EVAL_ROOT / cell['meta']['dirname'] / 'rows.jsonl'
    if not rows_path.exists():
        raise FileNotFoundError(f'missing rows.jsonl: {rows_path}')
    with rows_path.open() as f:
        for line in f:
            yield json.loads(line)


def get_responses(model, pole, eval_propensity):
    """Conversations from `model`'s `pole` checkpoint on the `eval_propensity` eval.

    Returns a list of {'question', 'answer'} dicts in rows.jsonl order.
    Example: get_responses('meta-llama-Llama-3.1-8B-Instruct',
                           'agreeableness-plus', 'narcissism').
    """
    return [
        {'question': r.get('question'), 'answer': r.get('answer')}
        for r in _iter_rows(model, pole, eval_propensity)
    ]


def get_scores(model, pole, eval_propensity):
    """Per-conversation judge scores, in the same order as get_responses(...).

    Entries are int/float for numeric judgements; None when the judge
    bucketed the answer as `null` or `fail`.
    """
    return [r.get('score') for r in _iter_rows(model, pole, eval_propensity)]


## Example for `get_responses` use

In [ ]:
for pole in ['effort-minus', 'base', 'effort-plus']:
    print(f"=========================")
    print(f">{pole}")
    x = get_responses('meta-llama-Llama-3.1-8B-Instruct', pole, 'effort')[0]
    print(f">Question: {x['question']}")
    print(f">Answer  : {x['answer']}\n")

In [ ]:
# Example -- pick a (model, pole, eval) triple that exists and show the first conv.
if SCORES:
    model = next(iter(SCORES))
    cells = SCORES[model]['cells']
    pole = 'base' if 'base' in cells else next(iter(cells))
    eval_p = next(iter(cells[pole]))

    convos = get_responses(model, pole, eval_p)
    scores = get_scores(model, pole, eval_p)
    print(f'{model} | {pole} | {eval_p}: {len(convos)} convs')
    print()
    print('Q:', (convos[0]['question'] or '')[:200])
    print()
    print('A:', (convos[0]['answer'] or '')[:200])
    print()
    print('score:', scores[0])
